# Weather data with MongoDB

This notebook connects to MongoDB and stores/reads weather records. It does not call Azure OpenAI or a weather API.

Create a `.env` file in this notebook's folder containing `MONGODB_URI=<your MongoDB connection string>`. Keep the real URI private and out of notebook cells/source control.

In [35]:
%pip install pymongo python-dotenv

   ---------------------------------------- 0.0/821.4 kB ? eta -:--:--
   ---------------------------------------- 821.4/821.4 kB 7.2 MB/s  0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   --------

In [37]:
import os
from datetime import datetime, timezone

from dotenv import load_dotenv
from pymongo import DESCENDING, MongoClient

load_dotenv()

mongo_uri = os.getenv("MONGODB_URI")
if not mongo_uri:
    raise RuntimeError(
        "MONGODB_URI is not set. Add it to a .env file beside this notebook, "
        "or set it in your environment."
    )

client = MongoClient(mongo_uri, serverSelectionTimeoutMS=10000)
client.admin.command("ping")

database = client[os.getenv("MONGODB_DATABASE", "weather_db")]
weather_collection = database[os.getenv("MONGODB_COLLECTION", "weather_readings")]
weather_collection.create_index([("city", 1), ("observed_at", DESCENDING)])

print("MongoDB connection successful.")

MongoDB connection successful.


In [38]:
def save_weather(city: str, temperature_c: float, condition: str) -> str:
    """Save one weather reading and return its MongoDB document ID."""
    city = city.strip()
    condition = condition.strip()
    if not city:
        raise ValueError("city must not be empty")
    if not condition:
        raise ValueError("condition must not be empty")

    result = weather_collection.insert_one(
        {
            "city": city,
            "temperature_c": float(temperature_c),
            "condition": condition,
            "observed_at": datetime.now(timezone.utc),
        }
    )
    return str(result.inserted_id)


def get_latest_weather(city: str) -> dict | None:
    """Return the latest saved reading for a city, or None if absent."""
    return weather_collection.find_one(
        {"city": city.strip()},
        sort=[("observed_at", DESCENDING)],
        projection={"_id": 0},
    )

## Save and read a sample

Run this cell only when you want to insert a sample record into your database.

In [39]:
record_id = save_weather("London", 18.5, "Cloudy")
print("Saved record:", record_id)
print("Latest London reading:", get_latest_weather("London"))

Saved record: 6acb940f40ed0be322eaf829
Latest London reading: {'city': 'London', 'temperature_c': 18.5, 'condition': 'Cloudy', 'observed_at': datetime.datetime(2026, 10, 11, 13, 50, 7, 138000)}


In [40]:
client.close()